# 5DATA001C.2 Machine Learning and Data Mining
## Final Python Notebook 3: Ensemble Classifier & Regression Decision Trees


**Student Name** : Diyapaththugama Vidanelage Sedani Lesara Sethumlee

**Student ID** : 20231352

**Code Peer Reviewer** : Upuli Maheshika

**Peer Review Date** : 26/03/2026

---
# PART 1 — Voting Ensemble Classifier for Loan Approval Status
---
## Step 1 — Import Libraries
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 1**

In [ ]:
# import pandas for loading and working with the dataset
import pandas as pd

---
## Step 2 — Load the Classification Dataset
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 2**

In [ ]:
# load the cleaned classification dataset prepared in Notebook 1
data = pd.read_csv('dataset_classification.csv')

---
## Step 3 — Inspect the Dataset
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 3**

In [ ]:
# display a sample of rows to confirm the dataset loaded correctly
data.head()

---
## Step 4 — Check Data Types
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 4**

In [ ]:
# check data types and non-null counts for all variables
data.info()

---
## Step 5 — Declare Input Features and Target Variable
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 5**

In [ ]:
# declare all input feature column names
feature_cols = ['age', 'income', 'home_ownership', 'emplyment_length', 'loan_intent',
                # continuing feature list
                'loan_amount', 'loan_interest_rate', 'payment_default_on_file', 'credit_history_length']
# assign all input features to X
X = data[feature_cols]  # Features
# assign the target variable to y
y = data['loan_approval_status']  # Target
# confirm X shape
print('X shape:', X.shape)
# display target class distribution
print('y distribution:', y.value_counts().to_dict())

---
## Step 5b — Apply MinMax Scaling
**Leveraged and Reused from: Seminar Sessions**

In [ ]:
# import MinMaxScaler to scale all numeric features to range [0,1]
from sklearn.preprocessing import MinMaxScaler
# instantiate the scaler
scaler = MinMaxScaler()
# apply fit_transform and store scaled features back in a dataframe
X = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

---
## Step 6 — Import Train-Test Split and Split the Dataset
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 6 & Prompt 7**

In [ ]:
# import train_test_split to sample training and test subsets
from sklearn.model_selection import train_test_split
# split into 80% training and 20% test with stratification and fixed random seed
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# confirm the split sizes
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

---
## Step 7 — Instantiate the Two Base Learners
**Leveraged and Reused from: Seminar Sessions**

NB and LR are selected as the two base learners for the ensemble.
LR uses the best hyperparameters found by GridSearchCV in Notebook 2 (C=10, solver=liblinear).
NB has the highest Recall (0.46) and LR has the highest AUC-ROC (0.8650) and Precision (0.76).
Combining them in a soft voting ensemble leverages NB's sensitivity with LR's precision.

In [ ]:
# import GaussianNB for Naive Bayes base learner
from sklearn.naive_bayes import GaussianNB
# import LogisticRegression for LR base learner
from sklearn.linear_model import LogisticRegression
# instantiate the NB base learner
nb = GaussianNB()
# instantiate the LR base learner using the best hyperparameters from GridSearchCV in Notebook 2
logreg = LogisticRegression(random_state=42, max_iter=1000, C=10, solver='liblinear')

---
## Step 8 — Import VotingClassifier and Declare the Ensemble
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 8 & Prompt 9**



In [ ]:
# import VotingClassifier from sklearn ensemble module
from sklearn.ensemble import VotingClassifier
# declare the list of base learners to combine into the ensemble
base_learners = [('nb', nb), ('logreg', logreg)]
# create the soft voting ensemble classifier using the two base learners
ensemble_learner = VotingClassifier(base_learners, voting='soft')

---
## Step 9 — Train the Ensemble Learner
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 10**

In [ ]:
# fit the voting ensemble classifier on the training subset
ensemble_learner.fit(X_train, y_train)

---
## Step 10 — Make Predictions with the Ensemble
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 11**

In [ ]:
# use the ensemble learner to predict loan approval status on the unseen test subset
y_pred_ensemble_learner = ensemble_learner.predict(X_test)

---
## Step 11 — Ensemble Accuracy Score
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 12**

In [ ]:
# import accuracy_score to evaluate ensemble performance
from sklearn.metrics import accuracy_score
# calculate the ensemble test accuracy
ensemble_learner_accuracy = accuracy_score(y_test, y_pred_ensemble_learner)
# print the ensemble accuracy
print('The voting ensemble classifier accuracy is: ', ensemble_learner_accuracy)

---
## Step 12 — Ensemble Confusion Matrix
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 13**



In [ ]:
# import confusion matrix module
from sklearn.metrics import confusion_matrix
# import ConfusionMatrixDisplay for plotting
from sklearn.metrics import ConfusionMatrixDisplay
# calculate the test confusion matrix for the ensemble learner
ensemble_learner_cm_test = confusion_matrix(y_test, y_pred_ensemble_learner, labels=ensemble_learner.classes_)
# create the confusion matrix display object
ensemble_learner_disp = ConfusionMatrixDisplay(ensemble_learner_cm_test, display_labels=ensemble_learner.classes_)
# plot the confusion matrix
ensemble_learner_disp.plot()
# set the title of the confusion matrix plot
ensemble_learner_disp.ax_.set_title('Ensemble Learner Confusion Matrix')

---
## Step 13 — Ensemble Classification Report
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 14**



In [ ]:
# import classification_report and print full metrics for the ensemble learner
from sklearn.metrics import classification_report
# print the ensemble learner classification report
print('Ensemble Learner Classification Report \n', classification_report(y_test, y_pred_ensemble_learner))

---
## Step 14 — Ensemble AUC-ROC Curve
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 15**



In [ ]:
# import RocCurveDisplay and plot the AUC-ROC curve for the ensemble learner
from sklearn.metrics import RocCurveDisplay
# plot the AUC-ROC curve using the ensemble model and test data
ensemble_learner = RocCurveDisplay.from_estimator(ensemble_learner, X_test, y_test)

---
## Step 14b — NB Base Learner Individual Results (for comparison)
**Leveraged and Reused from: Seminar Sessions**


In [ ]:
# re-train NB individually to get its standalone metrics for comparison
nb_solo = GaussianNB()
# fit NB on training data
nb_solo.fit(X_train, y_train)
# predict on test data
y_pred_nb_solo = nb_solo.predict(X_test)
# print NB confusion matrix
cm_nb = confusion_matrix(y_test, y_pred_nb_solo, labels=nb_solo.classes_)
# display NB confusion matrix
disp_nb = ConfusionMatrixDisplay(cm_nb, display_labels=nb_solo.classes_)
# plot the NB confusion matrix
disp_nb.plot()
# set the title
disp_nb.ax_.set_title('NB Base Learner — Confusion Matrix')

**Leveraged and Reused from: Seminar Sessions**

In [ ]:
# print the NB classification report for comparison with the ensemble
print('NB Base Learner Classification Report \n', classification_report(y_test, y_pred_nb_solo))

**Leveraged and Reused from: Seminar Sessions**

In [ ]:
# plot the AUC-ROC curve for the NB base learner
nb_solo = RocCurveDisplay.from_estimator(nb_solo, X_test, y_test)

---
## Step 14c — LR Base Learner Individual Results (for comparison)
**Leveraged and Reused from: Seminar Sessions**


In [ ]:
# re-train LR individually with best hyperparameters to get its standalone metrics
lr_solo = LogisticRegression(random_state=42, max_iter=1000, C=10, solver='liblinear')
# fit LR on training data
lr_solo.fit(X_train, y_train)
# predict on test data
y_pred_lr_solo = lr_solo.predict(X_test)
# print LR confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr_solo, labels=lr_solo.classes_)
# display LR confusion matrix
disp_lr = ConfusionMatrixDisplay(cm_lr, display_labels=lr_solo.classes_)
# plot the LR confusion matrix
disp_lr.plot()
# set the title
disp_lr.ax_.set_title('LR Base Learner — Confusion Matrix')

**Leveraged and Reused from: Seminar Sessions**

In [ ]:
# print the LR classification report for comparison with the ensemble
print('LR Base Learner Classification Report \n', classification_report(y_test, y_pred_lr_solo))

**Leveraged and Reused from: Seminar Sessions**

In [ ]:
# plot the AUC-ROC curve for the LR base learner
lr_solo = RocCurveDisplay.from_estimator(lr_solo, X_test, y_test)

---
# PART 2 — Regression Decision Trees for Maximum Loan Amount
---
## Step 15 — Import Libraries for Regression
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 1**

In [ ]:
# import pandas for loading the regression dataset
import pandas as pd

---
## Step 16 — Load the Regression Dataset
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 2**

In [ ]:
# load the cleaned regression dataset (approved clients only) prepared in Notebook 1
dataset = pd.read_csv('dataset_regression.csv')

---
## Step 17 — Check Data Types
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 3**

In [ ]:
# check data types and non-null counts for all variables in the regression dataset
dataset.info()

---
## Step 18 — Declare Input Features and Target Variable for Regression
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 4**

In [ ]:
# declare input features X — all variables except the regression target
X = dataset.drop('max_allowed_loan', axis=1)
# declare target variable y — the maximum loan amount
y = dataset['max_allowed_loan']
# confirm X shape
print('X shape:', X.shape)
# confirm y shape
print('y shape:', y.shape)
# display all feature names
print('Features:', list(X.columns))

---
## Step 19 — Split into Training and Test Subsets
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 5**

In [ ]:
# import train_test_split for sampling
from sklearn.model_selection import train_test_split
# split into 80% training and 20% test with a fixed random seed
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# confirm the split sizes
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

---
## Step 20 — Import Decision Tree Regressor
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 6**

In [ ]:
# import DecisionTreeRegressor from sklearn tree module
from sklearn.tree import DecisionTreeRegressor

---
## Step 21 — Build DT-1: Fully Grown Regression Decision Tree
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 7**

In [ ]:
# instantiate a fully grown decision tree regressor with no depth limit
DT_regressor = DecisionTreeRegressor(random_state=42)
# train DT-1 on the training subset
DT_regressor.fit(X_train, y_train)
# display tree depth
print('DT-1 depth:', DT_regressor.get_depth())
# display number of leaves
print('DT-1 leaves:', DT_regressor.get_n_leaves())

---
## Step 22 — DT-1 Predictions and Regression Metrics
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 8 & Prompt 9**



In [ ]:
# use DT-1 to predict maximum loan amount on the unseen test subset
y_pred = DT_regressor.predict(X_test)
# import sklearn metrics module for regression evaluation
from sklearn import metrics
# print the MAE Mean Absolute Error for DT-1
print('MAE:', metrics.mean_absolute_error(y_test, y_pred))
# print the MSE Mean Squared Error for DT-1
print('MSE:', metrics.mean_squared_error(y_test, y_pred))
# print the R-Squared score for DT-1
print('R2:', metrics.r2_score(y_test, y_pred))

---
## Step 23 — Visualise DT-1
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 10 & Prompt 11 & Prompt 12**



In [ ]:
from sklearn import tree
import matplotlib.pyplot as plt

# --- Plot DT-1 (Fully Grown Tree) ---
# Use a large figure size to fit the complex fully grown tree
plt.figure(figsize=(25, 12))
tree.plot_tree(DT_regressor, feature_names=X_train.columns.tolist(), filled=True, fontsize=6)
plt.title("DT-1: Fully Grown Regression Tree")
plt.show()

---
## Step 24 — Build DT-2: Pruned Regression Decision Tree (max_depth=4)
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 13**

In [ ]:
# instantiate a pruned decision tree regressor limited to a maximum depth of 4 levels
DT_regressor2 = DecisionTreeRegressor(max_depth=4, random_state=42)
# train DT-2 on the training subset
DT_regressor2.fit(X_train, y_train)
# predict maximum loan amount on the unseen test subset using DT-2
y_pred = DT_regressor2.predict(X_test)
# display the tree depth
print('DT-2 depth:', DT_regressor2.get_depth())
# display number of leaves
print('DT-2 leaves:', DT_regressor2.get_n_leaves())

---
## Step 25 — DT-2 Regression Metrics
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 9**



In [ ]:
# print the MAE Mean Absolute Error for DT-2
print('MAE:', metrics.mean_absolute_error(y_test, y_pred))
# print the MSE Mean Squared Error for DT-2
print('MSE:', metrics.mean_squared_error(y_test, y_pred))
# print the R-Squared score for DT-2
print('R2:', metrics.r2_score(y_test, y_pred))

---
## Step 26 — Visualise DT-2 (Pruned Tree)
**Leveraged and Reused from: Code Reuse Session 3 — Prompt 10 & Prompt 11 & Prompt 12**


In [ ]:
# create a figure for the pruned decision tree
Tree_figure2 = plt.figure(figsize=(25, 15))
# plot the pruned DT-2 with feature names and colour
DT_Graph2 = tree.plot_tree(DT_regressor2, feature_names=list(X_train.columns), filled=True)
# save the pruned tree as a high-resolution SVG image
Tree_figure2.savefig('DT2_pruned.svg')
# display the tree
plt.show()

---
## Step 27 — Predict Maximum Loan Amount for Client 60256
**Leveraged and Reused from: Seminar Sessions**

Client 60256 attributes: Age=56, Income=57000, Home Ownership=RENT (encoded=3), Employment=15, Loan Intent=MEDICAL (encoded=3), Loan Amount=25700, Interest Rate=23, Default=N (encoded=0), Credit History=35.


In [ ]:
# create a dataframe with Client 60256 attribute values for prediction
# home_ownership encoding: RENT=3 | loan_intent encoding: MEDICAL=3 | payment_default_on_file: N=0
client_60256 = pd.DataFrame(
    # define each attribute as a list with one value
    {
    # client age in years
    'age': [56],
    # client annual income in GBP
    'income': [57000],
    # home ownership encoded: RENT = 3
    'home_ownership': [3],
    # years of employment
    'emplyment_length': [15],
    # loan intent encoded: MEDICAL = 3
    'loan_intent': [3],
    # requested loan amount in GBP
    'loan_amount': [25700],
    # loan interest rate as percentage
    'loan_interest_rate': [23],
    # payment default on file encoded: N = 0
    'payment_default_on_file': [0],
    # credit history length in years
    'credit_history_length': [35]
# close the dictionary and DataFrame constructor
})
# use DT-2 (best model) to predict the maximum loan amount for client 60256
prediction = DT_regressor2.predict(client_60256)
# print the predicted maximum loan amount
print('Predicted Maximum Loan Amount for Client 60256: £', round(prediction[0], 2))